In [1]:
install.packages("sqldf")
install.packages("dplyr")
install.packages("ggplot2")

library(sqldf)
library(dplyr)
library(ggplot2)

print("Libraries loaded")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




[1] "Libraries loaded"


In [2]:
customers  <- read.csv("customers.csv")
drivers    <- read.csv("drivers.csv")
vehicles   <- read.csv("vehicles.csv")
hubs       <- read.csv("hubs.csv")
orders     <- read.csv("orders.csv")
deliveries <- read.csv("deliveries.csv")
incidents  <- read.csv("incidents.csv")
complaints <- read.csv("complaints.csv")
app_events <- read.csv("app_events.csv")

cat("All datasets loaded\n")
cat("Orders:", nrow(orders), "rows\n")
cat("Deliveries:", nrow(deliveries), "rows\n")
cat("Customers:", nrow(customers), "rows\n")

All datasets loaded
Orders: 1250 rows
Deliveries: 950 rows
Customers: 650 rows


In [3]:
# fixing zone names across all tables
zone_map <- c(
  "NORTH" = "North", "north" = "North",
  "SOUTH" = "South",
  "EAST" = "East",
  "WEST" = "West",
  "CENTRAL" = "Central", "Ctr" = "Central",
  "AIRPORT" = "Airport",
  "RiverSide" = "Riverside"
)

fix_zones <- function(df, cols) {
  for (col in cols) {
    df[[col]] <- ifelse(df[[col]] %in% names(zone_map),
                        zone_map[df[[col]]],
                        df[[col]])
  }
  return(df)
}

customers  <- fix_zones(customers,  c("home_zone"))
drivers    <- fix_zones(drivers,    c("base_zone"))
vehicles   <- fix_zones(vehicles,   c("assigned_zone"))
orders     <- fix_zones(orders,     c("pickup_zone", "dropoff_zone"))
app_events <- fix_zones(app_events, c("zone_context"))

cat("Zones cleaned\n")
cat("Unique zones:", paste(sort(unique(customers$home_zone)), collapse=", "), "\n")

Zones cleaned
Unique zones: Airport, Central, East, North, Riverside, South, West 


In [4]:
query1 <- sqldf("
  SELECT
    o.pickup_zone,
    COUNT(*) as total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'OnTime' THEN 1 ELSE 0 END) as on_time,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) as delayed,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) as failed,
    ROUND(SUM(CASE WHEN d.delivery_status = 'OnTime' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as on_time_pct
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY on_time_pct ASC
")

print(query1)

  pickup_zone total_deliveries on_time delayed failed on_time_pct
1     Central              174      90      51     33        51.7
2     Airport              113      70      31     12        61.9
3   Riverside              119      76      25     18        63.9
4        East              156     106      31     19        67.9
5       North              135      92      21     22        68.1
6        West              114      79      21     14        69.3
7       South              139     103      22     14        74.1


In [5]:
query2 <- sqldf("
  SELECT
    c.customer_type,
    COUNT(DISTINCT c.customer_id) as total_customers,
    COUNT(DISTINCT co.complaint_id) as total_complaints,
    ROUND(COUNT(DISTINCT co.complaint_id) * 100.0 / COUNT(DISTINCT c.customer_id), 1) as complaint_rate_pct
  FROM customers c
  LEFT JOIN complaints co ON c.customer_id = co.customer_id
  GROUP BY c.customer_type
  ORDER BY complaint_rate_pct DESC
")

print(query2)

  customer_type total_customers total_complaints complaint_rate_pct
1    Enterprise              50               28               56.0
2      Consumer             476              242               50.8
3           SME             124               50               40.3


In [6]:
query3 <- sqldf("
  SELECT
    h.hub_name,
    h.zone,
    h.hub_type,
    COUNT(d.delivery_id) as total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) as failed_deliveries,
    ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(d.delivery_id), 1) as failure_rate_pct
  FROM hubs h
  LEFT JOIN deliveries d ON h.hub_id = d.hub_id
  GROUP BY h.hub_name, h.zone, h.hub_type
  ORDER BY failure_rate_pct DESC
")

print(query3)

        hub_name      zone  hub_type total_deliveries failed_deliveries
1  Midtown Relay   Central  Charging              128                26
2   Central Core   Central   Control              115                23
3    Airport Hub   Airport  Dispatch              104                15
4      West Gate      West  Dispatch              127                16
5 North Exchange     North  Dispatch              136                17
6  Riverside Hub Riverside Warehouse              115                14
7     South Link     South  Dispatch              106                10
8      East Dock      East Warehouse              119                11
  failure_rate_pct
1             20.3
2             20.0
3             14.4
4             12.6
5             12.5
6             12.2
7              9.4
8              9.2


In [7]:
query4 <- sqldf("
  SELECT
    dr.driver_id,
    dr.driver_rating,
    dr.years_experience,
    COUNT(d.delivery_id) as total_deliveries,
    SUM(d.manual_route_override_count) as total_overrides,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) as failed_deliveries,
    ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(d.delivery_id), 1) as failure_rate_pct
  FROM drivers dr
  JOIN deliveries d ON dr.driver_id = d.driver_id
  GROUP BY dr.driver_id, dr.driver_rating, dr.years_experience
  HAVING COUNT(d.delivery_id) >= 5
  ORDER BY total_overrides DESC
  LIMIT 10
")

print(query4)

   driver_id driver_rating years_experience total_deliveries total_overrides
1       D127          4.19               10                6              17
2       D087          4.43               13               12              16
3       D130          3.64                8                8              16
4       D108          4.33               10               11              15
5       D131          4.26                9                9              15
6       D069          5.00                2                7              14
7       D105          3.71                2                7              14
8       D017          4.34                1               10              13
9       D028          4.07               11                7              13
10      D008          3.88                9                8              12
   failed_deliveries failure_rate_pct
1                  0              0.0
2                  2             16.7
3                  1             12.5
4

In [8]:
query5 <- sqldf("
  SELECT
    v.maintenance_status,
    COUNT(DISTINCT v.vehicle_id) as total_vehicles,
    COUNT(i.incident_id) as total_incidents,
    ROUND(COUNT(i.incident_id) * 1.0 / COUNT(DISTINCT v.vehicle_id), 1) as incidents_per_vehicle
  FROM vehicles v
  LEFT JOIN deliveries d ON v.vehicle_id = d.vehicle_id
  LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
  GROUP BY v.maintenance_status
  ORDER BY incidents_per_vehicle DESC
")

print(query5)

  maintenance_status total_vehicles total_incidents incidents_per_vehicle
1          Scheduled             17              46                   2.7
2           InRepair             36              84                   2.3
3             Active             67             150                   2.2
